# Getting started with ADK, MCP and A2A

Companion notebook to the guide [Getting started with ADK, MCP and A2A](https://tahreemrasul.github.io/adk-mcp-a2a-session/) by Tahreem Rasul. The guide explains the concepts; this notebook holds the code.

One currency-conversion function runs through every part:

| Part | The function is used as | Guide section |
|---|---|---|
| 1 | a tool, through function calling by hand | [4. Function calling](https://tahreemrasul.github.io/adk-mcp-a2a-session/#function-calling) |
| 2 | a tool of an ADK agent | [6. ADK](https://tahreemrasul.github.io/adk-mcp-a2a-session/#adk) |
| 3 | a tool on an MCP server | [7. MCP](https://tahreemrasul.github.io/adk-mcp-a2a-session/#mcp) |
| 4 | part of an agent that another agent calls over A2A | [8. A2A](https://tahreemrasul.github.io/adk-mcp-a2a-session/#a2a) |

**Before running:** get a free Gemini API key from [Google AI Studio](https://aistudio.google.com/apikey). In Colab, click the key icon in the left sidebar, add a secret named `GOOGLE_API_KEY`, and turn on notebook access. Then run the cells from top to bottom.

## 0. Setup

In [ ]:
%pip install -q "google-adk[a2a]==2.15.0" "mcp==2.3.0" "uvicorn==0.54.0"

If Colab asks to restart the session after the install, restart it and continue with the next cell.

In [ ]:
import logging, os, warnings

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except ImportError:  # running outside Colab: set GOOGLE_API_KEY in the environment first
    pass

MODEL = "gemini-flash-latest"

warnings.filterwarnings("ignore")  # keep the output readable
logging.disable(logging.WARNING)

## 1. Function calling by hand

A tool starts as an ordinary function. The model reads its name, type hints and docstring to decide when to call it. The rates are made up and fixed, so every run gives the same numbers.

In [ ]:
RATES_PER_EUR = {"EUR": 1.0, "USD": 1.08, "GBP": 0.85, "JPY": 162.0}

def get_exchange_rate(base: str, target: str) -> float:
    """Return how many units of `target` currency one unit of `base` buys.

    Args:
        base: ISO currency code to convert from, e.g. "EUR".
        target: ISO currency code to convert to, e.g. "JPY".
    """
    return round(RATES_PER_EUR[target.upper()] / RATES_PER_EUR[base.upper()], 4)

Ask a question and pass the function as a tool. Automatic function calling is switched off, so the reply is what the model actually returns: **a request to call the function, not an answer.**

In [ ]:
from google import genai
from google.genai import types

chat = genai.Client().chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(
        tools=[get_exchange_rate],
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    ),
)

response = chat.send_message("How many yen is 50 euros?")
call = response.function_calls[0]
print("The model asks to call:", call.name, dict(call.args))

The model never runs the function itself. Run it, send the result back, and the model writes the answer:

In [ ]:
result = get_exchange_rate(**call.args)
response = chat.send_message(
    types.Part.from_function_response(name=call.name, response={"result": result})
)
print(response.text)

## 2. An ADK agent

ADK runs the loop from [section 5 of the guide](https://tahreemrasul.github.io/adk-mcp-a2a-session/#agents): it sends the question, runs every tool the model asks for, returns the results, and repeats until the model answers.

The same agent is used in Parts 2, 3 and 4. Only its tools change, so it is created by one function:

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

def make_currency_agent(tools):
    return Agent(
        name="currency_agent",
        model=MODEL,
        description="Converts amounts between currencies.",
        instruction="You convert amounts between currencies. Always use get_exchange_rate; never guess a rate.",
        tools=tools,
    )

A small helper sends a question and prints each tool call and the final answer:

In [ ]:
from IPython.display import Markdown, display

async def ask(runner, question):
    events = await runner.run_debug(question, quiet=True)
    answers = []
    for event in events:
        for call in event.get_function_calls():
            print(f"🔧 {event.author} calls {call.name}({call.args})")
        for result in event.get_function_responses():
            # MCP tools wrap the value in extra fields; show just the value
            print(f"   ↳ {result.response.get('structuredContent', result.response)}")
        if event.is_final_response() and event.content and event.content.parts:
            text = "".join(part.text or "" for part in event.content.parts).strip()
            if text and text not in answers:
                answers.append(text)
                display(Markdown(f"**{event.author}:** {text}"))

The question needs two conversions. The agent decides by itself to call the tool twice:

In [ ]:
runner = InMemoryRunner(agent=make_currency_agent(tools=[get_exchange_rate]))
await ask(runner, "I have 200 USD. How much is that in GBP and in JPY?")

The runner keeps a **session**, so a follow-up question can rely on the earlier one:

In [ ]:
await ask(runner, "And in euros?")

## 3. Move the tool to an MCP server

The tool currently exists only inside this notebook. Putting it behind an **MCP server** lets any MCP application use it, such as Claude Desktop, VS Code or another agent.

The next cell saves a complete MCP server to a file. (In `mcp` 2.x the class is `MCPServer`; older tutorials call it `FastMCP`.)

In [ ]:
%%writefile currency_server.py
from mcp.server.mcpserver import MCPServer

mcp = MCPServer("currency")

RATES_PER_EUR = {"EUR": 1.0, "USD": 1.08, "GBP": 0.85, "JPY": 162.0}

@mcp.tool()
def get_exchange_rate(base: str, target: str) -> float:
    """Return how many units of `target` currency one unit of `base` buys."""
    return round(RATES_PER_EUR[target.upper()] / RATES_PER_EUR[base.upper()], 4)

if __name__ == "__main__":
    mcp.run()  # stdio: the client starts this script and talks to it over stdin/stdout

ADK connects as an MCP client: it starts the server as a subprocess and asks which tools it offers.

In [ ]:
import sys
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

SERVER_PATH = os.path.abspath("currency_server.py")
if not os.path.exists(SERVER_PATH):
    raise FileNotFoundError("currency_server.py is missing. Run the %%writefile cell above first.")

def currency_mcp_tools():
    return McpToolset(
        connection_params=StdioConnectionParams(
            server_params=StdioServerParameters(command=sys.executable, args=[SERVER_PATH]),
            timeout=10,
        )
    )

mcp_tools = currency_mcp_tools()
for tool in await mcp_tools.get_tools():
    print(tool.name, "-", tool.description)

The agent is the same as in Part 2. Only the tools argument changes.

> **Before running:** the function now runs in a separate process. Will the answer change?

In [ ]:
runner = InMemoryRunner(agent=make_currency_agent(tools=[mcp_tools]))
await ask(runner, "I have 200 USD. How much is that in GBP and in JPY?")

## 4. Share the agent with A2A

A trip-planning agent, built by another team, needs currency conversion. Instead of re-implementing it, it hands the task to our currency agent over A2A.

`to_a2a()` turns the currency agent into a web app that serves an **agent card**. It runs in the background on port 8001 and still gets its rates from the MCP server.

In [ ]:
import threading, time, uvicorn
from google.adk.a2a.utils.agent_to_a2a import to_a2a

a2a_app = to_a2a(make_currency_agent(tools=[currency_mcp_tools()]), port=8001)
server = uvicorn.Server(uvicorn.Config(a2a_app, host="localhost", port=8001, log_level="error"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)

The agent card tells other agents what this agent does and where to reach it:

In [ ]:
import httpx, json
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent, AGENT_CARD_WELL_KNOWN_PATH

CARD_URL = "http://localhost:8001" + AGENT_CARD_WELL_KNOWN_PATH
print(json.dumps(httpx.get(CARD_URL).json(), indent=2))

The trip planner has its own hotel-price tool and reaches the currency agent through `RemoteA2aAgent`, which needs only the card URL:

In [ ]:
HOTEL_NIGHT = {"Tokyo": ("JPY", 18000), "London": ("GBP", 140), "New York": ("USD", 210)}  # made-up prices

def get_hotel_price(city: str) -> dict:
    """Return the average price of one hotel night in `city`, in the local currency."""
    currency, price = HOTEL_NIGHT[city.title()]
    return {"currency": currency, "price_per_night": price}

trip_agent = Agent(
    name="trip_planner",
    model=MODEL,
    description="Plans trip budgets.",
    instruction=(
        "You help people budget trips. Look up hotel prices with get_hotel_price. "
        "For any currency conversion, transfer to currency_agent."
    ),
    tools=[get_hotel_price],
    sub_agents=[RemoteA2aAgent(name="currency_agent", description="Converts amounts between currencies.", agent_card=CARD_URL)],
)

runner = InMemoryRunner(agent=trip_agent)
await ask(runner, "I'm staying 3 nights in Tokyo. What will the hotel cost in euros?")

The output shows `trip_planner` calling its own tool, then transferring to `currency_agent`, which answers from its own server.

## Clean up

In [ ]:
server.should_exit = True
await mcp_tools.close()

## Further reading

- [ADK documentation](https://adk.dev/): build agents locally and test them in the `adk web` UI
- [MCP documentation](https://modelcontextprotocol.io/docs/getting-started/intro)
- [A2A protocol](https://a2a-protocol.org/latest/) and [A2A in ADK](https://adk.dev/a2a/intro/)
- [The guide](https://tahreemrasul.github.io/adk-mcp-a2a-session/)